# Advanced Methods
Trying out a new NLP pipeline I adapted from different kaggle problems, and here I'll use methods like **logistic regression**, **naive bayes classifiers**, **support-vector-machines**, **xgboost** and even **deep learning**.

In [1]:
# libraries for part 1
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
import re
import sys

pd.set_option('display.max_columns', 200)
plt.style.use("ggplot")

# Libraries for part 2
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import scipy.sparse
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Libraries for part 3
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.decomposition import TruncatedSVD

import gc; gc.enable()
from IPython.display import clear_output

In [2]:
# read in data
df = pl.read_csv("./data/995,000_rows_preprocessed.csv", n_rows=25_000).to_pandas()
df = df[['type', 'content']].rename(columns={'type':'isfake'}) # rename col for binary

real_labels = ['reliable']
fake_labels = ['bias','fake','unreliable','rumor','conspiracy','clickbait','junksci','satire','hate','political']
drop_labels = ['unknown','nan','2018-02-10 13:43:39.521661']

df = df.loc[~df['isfake'].isin(drop_labels)].iloc[:20_000] # limit to 20k samples
df['isfake'] = df['isfake'].map(lambda x : 0 if x in real_labels else 1)

x, y = df['content'], df['isfake']
x_train, x_tmp, y_train, y_tmp = train_test_split(x, y, test_size=0.2, stratify=y, random_state=1000)
x_val, x_test, y_val, y_test = train_test_split(x_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=1000)

print(x_train.shape)
print(x_val.shape)
print(x_test.shape)

del df, x, y
gc.collect(); # we remove big dataframe to save RAM

(16000,)
(2000,)
(2000,)


In [3]:
# tfidf
tfv = TfidfVectorizer(min_df=3, max_features=int(1e4), strip_accents='unicode', analyzer='word', token_pattern=r'\w{1,}',
            ngram_range=(1,3), use_idf=True, smooth_idf=True, sublinear_tf=True, stop_words='english')

tfv.fit(list(x_train) + list(x_val) + list(x_test))
x_train_tfv = tfv.transform(x_train)
x_val_tfv = tfv.transform(x_val)
x_test_tfv = tfv.transform(x_test)

In [4]:
# logistic regression
smoter = SMOTE(sampling_strategy='minority', random_state=1000)
x_train_tfv_sm, y_train_sm = smoter.fit_resample(x_train_tfv, y_train)

clf = LogisticRegression(C=1.0, max_iter=1000)
clf.fit(x_train_tfv_sm, y_train_sm)
val_preds = clf.predict(x_val_tfv)
print('F1-Score: {}'.format(f1_score(y_val, val_preds)))
print(classification_report(y_val, val_preds))

del tfv, smoter, clf
gc.collect();

F1-Score: 0.9614262560777957
              precision    recall  f1-score   support

           0       0.90      0.84      0.87       473
           1       0.95      0.97      0.96      1527

    accuracy                           0.94      2000
   macro avg       0.93      0.91      0.92      2000
weighted avg       0.94      0.94      0.94      2000



In [5]:
# bag of words
ctv = CountVectorizer(max_features=int(1e4), analyzer='word', token_pattern=r'\w{1,}', ngram_range=(1,3), stop_words='english')

ctv.fit(list(x_train) + list(x_val) + list(x_test))
x_train_ctv = ctv.transform(x_train)
x_val_ctv = ctv.transform(x_val)
x_test_ctv = ctv.transform(x_test)

In [6]:
smoter = SMOTE(sampling_strategy='minority', random_state=1000)
x_train_ctv_sm, y_train_sm = smoter.fit_resample(x_train_ctv, y_train)

clf = LogisticRegression(C=1.0, max_iter=1000)
clf.fit(x_train_ctv_sm, y_train_sm)
val_preds = clf.predict(x_val_ctv)
print('F1-Score: {}'.format(f1_score(y_val, val_preds)))
print(classification_report(y_val, val_preds))

del ctv, smoter, clf
gc.collect();

F1-Score: 0.9290236587804065
              precision    recall  f1-score   support

           0       0.75      0.83      0.79       473
           1       0.95      0.91      0.93      1527

    accuracy                           0.89      2000
   macro avg       0.85      0.87      0.86      2000
weighted avg       0.90      0.89      0.90      2000



In [7]:
# naive bayes
clf = MultinomialNB()
clf.fit(x_train_tfv_sm, y_train_sm)
val_preds = clf.predict(x_val_tfv)
print('F1-Score: {}'.format(f1_score(y_val, val_preds)))
print(classification_report(y_val, val_preds))

del clf
gc.collect();

F1-Score: 0.9224461906388794
              precision    recall  f1-score   support

           0       0.70      0.89      0.79       473
           1       0.96      0.88      0.92      1527

    accuracy                           0.89      2000
   macro avg       0.83      0.89      0.86      2000
weighted avg       0.90      0.89      0.89      2000



In [8]:
# naive bayes
clf = MultinomialNB()
clf.fit(x_train_ctv_sm, y_train_sm)
val_preds = clf.predict(x_val_ctv)
print('F1-Score: {}'.format(f1_score(y_val, val_preds)))
print(classification_report(y_val, val_preds))

del clf
gc.collect();

F1-Score: 0.936115569823435
              precision    recall  f1-score   support

           0       0.83      0.73      0.78       473
           1       0.92      0.95      0.94      1527

    accuracy                           0.90      2000
   macro avg       0.88      0.84      0.86      2000
weighted avg       0.90      0.90      0.90      2000



In [9]:
# xgboost
clf = XGBClassifier(random_state=1000)
clf.fit(x_train_tfv_sm, y_train_sm)
val_preds = clf.predict(x_val_tfv)
print('F1-Score: {}'.format(f1_score(y_val, val_preds)))
print(classification_report(y_val, val_preds))

del clf
gc.collect();

F1-Score: 0.959845807902345
              precision    recall  f1-score   support

           0       0.92      0.81      0.86       473
           1       0.94      0.98      0.96      1527

    accuracy                           0.94      2000
   macro avg       0.93      0.89      0.91      2000
weighted avg       0.94      0.94      0.94      2000



In [10]:
# xgboost
clf = XGBClassifier(random_state=1000)
clf.fit(x_train_ctv_sm, y_train_sm)
val_preds = clf.predict(x_val_ctv)
print('F1-Score: {}'.format(f1_score(y_val, val_preds)))
print(classification_report(y_val, val_preds))

del clf
gc.collect();

F1-Score: 0.9635782747603834
              precision    recall  f1-score   support

           0       0.95      0.80      0.87       473
           1       0.94      0.99      0.96      1527

    accuracy                           0.94      2000
   macro avg       0.95      0.89      0.92      2000
weighted avg       0.94      0.94      0.94      2000



In [11]:
# support-vector-machine with singular-value-decomposition and z-scaling | TF-IDF
svd = TruncatedSVD(n_components=128)
svd.fit(x_train_tfv)
x_train_tfv_svd = svd.transform(x_train_tfv)
x_val_tfv_svd = svd.transform(x_val_tfv)

# z-scaling
scl = StandardScaler(with_mean=False)
scl.fit(x_train_tfv_svd)
x_train_tfv_svd_scl = scl.transform(x_train_tfv_svd)
x_val_tfv_svd_scl = scl.transform(x_val_tfv_svd)

In [12]:
# svm
smoter = SMOTE(sampling_strategy='minority', random_state=1000)
x_train_tfv_svd_scl_sm, y_train_sm = smoter.fit_resample(x_train_tfv_svd_scl, y_train)

clf = SVC(C=1.0)
clf.fit(x_train_tfv_svd_scl_sm, y_train_sm)
val_preds = clf.predict(x_val_tfv_svd_scl)
print('F1-Score: {}'.format(f1_score(y_val, val_preds)))
print(classification_report(y_val, val_preds))

del svd, scl, smoter, clf
gc.collect();

F1-Score: 0.9559500328731098
              precision    recall  f1-score   support

           0       0.85      0.87      0.86       473
           1       0.96      0.95      0.96      1527

    accuracy                           0.93      2000
   macro avg       0.90      0.91      0.91      2000
weighted avg       0.93      0.93      0.93      2000



In [13]:
# support-vector-machine with singular-value-decomposition | BOW
svd = TruncatedSVD(n_components=128)
svd.fit(x_train_ctv)
x_train_ctv_svd = svd.transform(x_train_ctv)
x_val_ctv_svd = svd.transform(x_val_ctv)

# z-scaling
scl = StandardScaler(with_mean=False)
scl.fit(x_train_ctv_svd)
x_train_ctv_svd_scl = scl.transform(x_train_ctv_svd)
x_val_ctv_svd_scl = scl.transform(x_val_ctv_svd)

In [14]:
# svm
smoter = SMOTE(sampling_strategy='minority', random_state=1000)
x_train_ctv_svd_scl_sm, y_train_sm = smoter.fit_resample(x_train_ctv_svd_scl, y_train)

clf = SVC(C=1.0)
clf.fit(x_train_ctv_svd_scl_sm, y_train_sm)
val_preds = clf.predict(x_val_ctv_svd_scl)
print('F1-Score: {}'.format(f1_score(y_val, val_preds)))
print(classification_report(y_val, val_preds))

del svd, scl, smoter, clf
gc.collect();

F1-Score: 0.9103585657370518
              precision    recall  f1-score   support

           0       0.70      0.76      0.73       473
           1       0.92      0.90      0.91      1527

    accuracy                           0.86      2000
   macro avg       0.81      0.83      0.82      2000
weighted avg       0.87      0.86      0.87      2000



<br><h3 style='text-align: center;'>Trying on test set</h3><br>

In [4]:
# logistic regression
smoter = SMOTE(sampling_strategy='minority', random_state=1000)
x_train_tfv_sm, y_train_sm = smoter.fit_resample(x_train_tfv, y_train)

clf = LogisticRegression(C=1.0, max_iter=1000)
clf.fit(x_train_tfv_sm, y_train_sm)
test_preds = clf.predict(x_test_tfv)
print('F1-Score: {}'.format(f1_score(y_test, test_preds)))
print(classification_report(y_test, test_preds))

del tfv, smoter, clf
gc.collect();

F1-Score: 0.9675324675324676
              precision    recall  f1-score   support

           0       0.92      0.87      0.89       472
           1       0.96      0.98      0.97      1528

    accuracy                           0.95      2000
   macro avg       0.94      0.92      0.93      2000
weighted avg       0.95      0.95      0.95      2000



In [16]:
# naive bayes
clf = MultinomialNB()
clf.fit(x_train_tfv_sm, y_train_sm)
test_preds = clf.predict(x_test_tfv)
print('F1-Score: {}'.format(f1_score(y_test, test_preds)))
print(classification_report(y_test, test_preds))

del clf
gc.collect();

F1-Score: 0.9173497267759563
              precision    recall  f1-score   support

           0       0.69      0.88      0.77       472
           1       0.96      0.88      0.92      1528

    accuracy                           0.88      2000
   macro avg       0.83      0.88      0.85      2000
weighted avg       0.90      0.88      0.88      2000



In [8]:
# naive bayes
clf = MultinomialNB()
clf.fit(x_train_ctv_sm, y_train_sm)
test_preds = clf.predict(x_test_ctv)
print('F1-Score: {}'.format(f1_score(y_test, test_preds)))
print(classification_report(y_test, test_preds))

del clf
gc.collect();

F1-Score: 0.9332901554404145
              precision    recall  f1-score   support

           0       0.80      0.75      0.77       472
           1       0.92      0.94      0.93      1528

    accuracy                           0.90      2000
   macro avg       0.86      0.85      0.85      2000
weighted avg       0.90      0.90      0.90      2000



In [10]:
# xgboost
clf = XGBClassifier(random_state=1000)
clf.fit(x_train_tfv_sm, y_train_sm)
test_preds = clf.predict(x_test_tfv)
print('F1-Score: {}'.format(f1_score(y_test, test_preds)))
print(classification_report(y_test, test_preds))

del clf
gc.collect();

F1-Score: 0.9634849455477258
              precision    recall  f1-score   support

           0       0.94      0.81      0.87       472
           1       0.94      0.98      0.96      1528

    accuracy                           0.94      2000
   macro avg       0.94      0.90      0.92      2000
weighted avg       0.94      0.94      0.94      2000



In [11]:
# xgboost
clf = XGBClassifier(random_state=1000)
clf.fit(x_train_ctv_sm, y_train_sm)
test_preds = clf.predict(x_test_ctv)
print('F1-Score: {}'.format(f1_score(y_test, test_preds)))
print(classification_report(y_test, test_preds))

del clf
gc.collect();

F1-Score: 0.9620898375278751
              precision    recall  f1-score   support

           0       0.95      0.79      0.86       472
           1       0.94      0.99      0.96      1528

    accuracy                           0.94      2000
   macro avg       0.95      0.89      0.91      2000
weighted avg       0.94      0.94      0.94      2000



In [12]:
# support-vector-machine with singular-value-decomposition and z-scaling | TF-IDF
svd = TruncatedSVD(n_components=128)
svd.fit(x_train_tfv)
x_train_tfv_svd = svd.transform(x_train_tfv)
x_test_tfv_svd = svd.transform(x_test_tfv)

# z-scaling
scl = StandardScaler(with_mean=False)
scl.fit(x_train_tfv_svd)
x_train_tfv_svd_scl = scl.transform(x_train_tfv_svd)
x_test_tfv_svd_scl = scl.transform(x_test_tfv_svd)

In [13]:
# svm
smoter = SMOTE(sampling_strategy='minority', random_state=1000)
x_train_tfv_svd_scl_sm, y_train_sm = smoter.fit_resample(x_train_tfv_svd_scl, y_train)

clf = SVC(C=1.0)
clf.fit(x_train_tfv_svd_scl_sm, y_train_sm)
test_preds = clf.predict(x_test_tfv_svd_scl)
print('F1-Score: {}'.format(f1_score(y_test, test_preds)))
print(classification_report(y_test, test_preds))

del svd, scl, smoter, clf
gc.collect();

F1-Score: 0.9562067830095489
              precision    recall  f1-score   support

           0       0.85      0.88      0.86       472
           1       0.96      0.95      0.96      1528

    accuracy                           0.93      2000
   macro avg       0.90      0.91      0.91      2000
weighted avg       0.93      0.93      0.93      2000



In [14]:
# support-vector-machine with singular-value-decomposition | BOW
svd = TruncatedSVD(n_components=128)
svd.fit(x_train_ctv)
x_train_ctv_svd = svd.transform(x_train_ctv)
x_test_ctv_svd = svd.transform(x_test_ctv)

# z-scaling
scl = StandardScaler(with_mean=False)
scl.fit(x_train_ctv_svd)
x_train_ctv_svd_scl = scl.transform(x_train_ctv_svd)
x_test_ctv_svd_scl = scl.transform(x_test_ctv_svd)

In [15]:
# svm
smoter = SMOTE(sampling_strategy='minority', random_state=1000)
x_train_ctv_svd_scl_sm, y_train_sm = smoter.fit_resample(x_train_ctv_svd_scl, y_train)

clf = SVC(C=1.0)
clf.fit(x_train_ctv_svd_scl_sm, y_train_sm)
test_preds = clf.predict(x_test_ctv_svd_scl)
print('F1-Score: {}'.format(f1_score(y_test, test_preds)))
print(classification_report(y_test, test_preds))

del svd, scl, smoter, clf
gc.collect();

F1-Score: 0.9166110740493663
              precision    recall  f1-score   support

           0       0.71      0.80      0.75       472
           1       0.93      0.90      0.92      1528

    accuracy                           0.88      2000
   macro avg       0.82      0.85      0.83      2000
weighted avg       0.88      0.88      0.88      2000

